# 04_2 — Special Core Analysis (SCAL), `CoreyRelPerm` & Static-to-Dynamic Log Blocking

This notebook demonstrates:
1. **Laboratory-to-Reservoir Core Corrections (`quick_pp.scal`)**: Overburden porosity/permeability stress compaction (`overburden_poroperm_correction`), Klinkenberg gas slippage correction (`klinkenberg_permeability_correction`), Dean-Stark clay-bound water correction (`dean_stark_cbw_correction`), and capillary pressure scaling (`pc_lab_to_reservoir`) using bundled COSTA MICP data (`notebooks/data/01_raw/COSTA/HW_core_data_all.csv`).
2. **Relative Permeability & Eclipse/Intersect `SWOF` Generation**: `CoreyRelPerm`, `corey_relperm`, `brooks_corey_relperm`, `let_relperm`, `fit_relperm`, and `generate_swof_table`.
3. **Static-to-Dynamic Log Blocking & Upscaling (`quick_pp.upscaling`)**: `block_logs` / `upscale_logs`, `cardwell_parsons_bounds`, `pore_volume_upscale`, `expand_blocked_logs`, and `plot_upscaled_logs`.

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
for _cand in (Path.cwd(), Path.cwd().parent):
    if (_cand / "quick_pp" / "__init__.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quick_pp import (
    CoreyRelPerm,
    corey_relperm,
    brooks_corey_relperm,
    let_relperm,
    fit_relperm,
    generate_swof_table,
    overburden_poroperm_correction,
    klinkenberg_permeability_correction,
    dean_stark_cbw_correction,
    pc_lab_to_reservoir,
    correct_core_data,
    block_logs,
    upscale_logs,
    cardwell_parsons_bounds,
    pore_volume_upscale,
    expand_blocked_logs,
    plot_upscaled_logs,
)

## Part A: Lab-to-Reservoir MICP Capillary Pressure & Overburden Core Corrections

In [ ]:
# Load bundled COSTA MICP & Core Plug dataset
core_csv = Path("data/01_raw/COSTA/HW_core_data_all.csv")
if not core_csv.exists():
    core_csv = Path("notebooks/data/01_raw/COSTA/HW_core_data_all.csv")

micp_df = pd.read_csv(core_csv)
sample_1 = micp_df[micp_df["Sample"] == 1].copy()
sample_1["CPORE"] = sample_1["Phi (frac.)"].clip(0.02, 0.40)
sample_1["CPERM"] = np.maximum(sample_1["K mD"], 15.0)

# Correct ambient porosity & air permeability to reservoir Net Confining Stress (NCS = 3500 psi)
poro_res, perm_res = overburden_poroperm_correction(
    sample_1["CPORE"],
    sample_1["CPERM"],
    net_confining_stress=3500.0,
    poro_compaction_coeff=4.0e-6,
    perm_compaction_coeff=3.0e-5,
    method="kozeny_carman",
)

# Convert laboratory Mercury-Air MICP (IFT=485 dynes/cm, theta=140 deg) to Reservoir Oil-Brine (IFT=30, theta=30)
sample_1["PC_RES_PSIA"] = pc_lab_to_reservoir(
    sample_1["O/B Pc (psia)"],
    ift_lab=485.0,
    theta_lab=140.0,
    ift_res=30.0,
    theta_res=30.0,
    poro_amb=sample_1["CPORE"],
    perm_amb=sample_1["CPERM"],
    poro_res=poro_res,
    perm_res=perm_res,
)

display(sample_1[["Sample", "RRT", "CPORE", "Equiv Brine Sat. (Frac.)", "O/B Pc (psia)", "PC_RES_PSIA"]].head(8).round(4))

## Part B: `CoreyRelPerm` Curve Fitting & Dynamic Simulator `SWOF` Handover

In [ ]:
# Instantiate, fit, and export Eclipse SWOF table using CoreyRelPerm
sw_pts = np.array([0.15, 0.22, 0.30, 0.38, 0.46, 0.54, 0.62, 0.70, 0.78])
krw_pts = np.array([0.000, 0.004, 0.018, 0.046, 0.092, 0.158, 0.245, 0.355, 0.480])
kro_pts = np.array([0.880, 0.670, 0.485, 0.330, 0.205, 0.112, 0.048, 0.012, 0.000])

relperm_estimator = CoreyRelPerm().fit(sw_pts, krw=krw_pts, kro=kro_pts, swirr=0.15, sor=0.22)
swof_bundle = relperm_estimator.to_swof(n_points=15, pc_entry=1.5, pc_exponent=1.6, pc_max=40.0)

print("Fitted CoreyRelPerm summary:", relperm_estimator.fit_result_)
display(swof_bundle["table"].head(10))

## Part C: Static-to-Dynamic Log Blocking (`block_logs`) & Cardwell-Parsons Anisotropy

In [ ]:
# Load HW-29 log data and perform cellular blocking + step-log expansion
data_path = Path("../quick_pp/machine_learning/mock_data/mock_data.parquet")
if not data_path.exists():
    data_path = Path("quick_pp/machine_learning/mock_data/mock_data.parquet")

log_df = pd.read_parquet(data_path).sort_values("DEPTH").reset_index(drop=True)
log_df["PHIT"] = log_df["PHIE"].clip(0.02, 0.38)
log_df["SWT"] = log_df["SW"].clip(0.05, 1.0)
log_df["VSHALE"] = ((log_df["GR"] - 15.0) / 110.0).clip(0.0, 1.0)
log_df["PERM"] = log_df["CORE_PERM"].fillna(10.0 ** (-1.0 + 12.0 * log_df["PHIT"])).clip(0.05, 5000.0)
log_df["ROCK_TYPE"] = np.where(log_df["PERM"] >= 100.0, "RT1_HIGH", np.where(log_df["PERM"] >= 10.0, "RT2_MOD", "RT3_LOW"))

# Upscale to 20-ft grid blocks using net-reservoir weighting (net_only=True)
cells_df = block_logs(
    log_df,
    cell_size=20.0,
    depth_col="DEPTH",
    vsh_col="VSHALE",
    phi_col="PHIT",
    phie_col="PHIE",
    sw_col="SWT",
    perm_col="PERM",
    facies_cols=["ROCK_TYPE"],
    perm_method="geometric",
    omega=0.33,
    net_only=True,
    cutoffs={"VSHALE": 0.45, "PHIT": 0.05, "SWT": 0.80},
)

# Expand blocked cells back onto fine-scale depth grid as step curves (_BLK suffix)
expanded_df = expand_blocked_logs(log_df, cells_df, depth_col="DEPTH")
display(cells_df[["CELL_ID", "TOP", "BOTTOM", "NTG", "PHIT", "SWT", "BVHC", "PERM_HARM", "PERM_GEOM", "PERM_ARITH", "KV_KH_RATIO", "ROCK_TYPE"]].head(8).round(4))

fig = plot_upscaled_logs(log_df, cells_df, depth_col="DEPTH", vsh_col="VSHALE", phi_col="PHIT", sw_col="SWT", perm_col="PERM")
fig.show()